[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.9 MB/s eta 0:00:00


In [2]:
import torch
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    bs, seq, dim = q.shape

    # Compute frequency
    # with respect to ermbedding dimensions
    # theta_i = 10000^{-2i/D}, where i in {0, 1, ..., D/2 - 1}
    # the larger is i, the smaller is the freq (theta)
    # higher freq (theta) detects local neighbors as it rotates faster
    # lower freq (theta) detects distant neighbors as it rotates slower
    dim_idx = torch.arange(0, dim, 2) # 2i
    theta = 1 / (10000 ** (dim_idx / dim))

    # Compute position 
    # position of the word in a sequence
    m = torch.arange(seq)

    # position angles
    # m * theta^T
    angles = torch.outer(m, theta) # seq, dim/2
    
    # Split into even/odd pairs
    cos = angles.cos().repeat_interleave(2, dim=-1).unsqueeze(0) # 1, seq, dim
    sin = angles.sin().repeat_interleave(2, dim=-1).unsqueeze(0)

    # Apply rotation
    def rotate_half(x):
        x1 = x[..., 0::2]
        x2 = x[..., 1::2]
        # [-x_1, x_0, -x_3, x_2, ...]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)

    return q_rot, k_rot

In [4]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (4.5ms)
  ✅ [2/4] Preserves norm (10.5ms)
  ✅ [3/4] Relative position property (19.0ms)
  ✅ [4/4] Gradient flow (15.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (49.2ms total)
  Progress saved. Run status() to see your dashboard.

